# Multimodal RAG

**Module:** 15 — VLMs & Multimodal

Retrieve over images, pages, and text — approaches, indexing, and failure modes.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Compare caption-then-RAG, dual-encoder, and hybrid page RAG
- Design indexes with text, captions, vectors, and ACL metadata
- Pack retrieved pages into VLM prompts with citations
- Audit configs for caption drift and layout loss


## Approaches

### Definition
Multimodal RAG retrieves visual/textual evidence and conditions a generator (often a VLM) on it.

### Why it matters
Text-only RAG misses charts/photos; stuffing all pages does not scale.

### How it works
Patterns: caption/OCR→text RAG; image dual-encoder; page MM embeddings; late interaction over patches.

### Intuition
RAG = show the model the right page — multimodal when the page is visual.

### Pitfalls
- Caption-only indexes omitting chart numbers
- Retrieve without attaching images
- Cross-tenant vector leakage

### When to use
Manuals with diagrams, slides, catalogs, scanned archives.


### Approach comparison

| Approach | Index | Generator sees | Risk |
|----------|-------|----------------|------|
| Caption→text | Captions/OCR | Text | Lost visuals |
| Image dual-encoder | Image vectors | Top images | Weak tiny text |
| Page MM embed | Page vectors | Page images+text | Storage |
| Hybrid | Text+image+meta | Best of both | Tuning |

```mermaid
flowchart TD
  Q[Question] --> R[Retriever]
  R --> T[(Text idx)]
  R --> V[(Visual idx)]
  T --> M[Merge/rerank]
  V --> M --> G[VLM] --> A[Cited answer]
```


In [ ]:
# Demo 1: caption-then-RAG index
from dataclasses import dataclass

@dataclass
class Page:
    page_id: str; caption: str; ocr: str; image_uri: str

PAGES = [
    Page("p1","Bar chart quarterly revenue","Q1 10 Q2 12 Q3 18 Q4 14","s3://img/p1.png"),
    Page("p2","Factory floor","safety first","s3://img/p2.png"),
    Page("p3","Invoice sample","Total 19.99 USD Invoice 1042","s3://img/p3.png"),
]
def retrieve(q,k=2):
    qw=set(q.lower().split())
    scored=sorted(((len(qw & set((p.caption+" "+p.ocr).lower().split())),p) for p in PAGES), reverse=True)
    return [p for s,p in scored[:k] if s>0]
for h in retrieve("What was Q3 revenue on the chart?"):
    print(h.page_id, h.ocr)


In [ ]:
# Demo 2: pack VLM RAG prompt
import json
def pack(question, pages):
    content = [{"type":"text","text":f"Answer using only attached pages. Cite page_id.\nQ: {question}"}]
    for p in pages:
        content += [{"type":"text","text":f"[page_id={p.page_id}] OCR: {p.ocr[:80]}"},
                    {"type":"image_url","image_url":{"url":p.image_uri}}]
    return {"model":"gpt-4o","messages":[{"role":"user","content":content}],"max_tokens":400}
print(json.dumps(pack("What was Q3 revenue?", retrieve("Q3 revenue chart")), indent=2)[:500])


## Indexing Strategy

### Definition
Indexing decides retrievability: captions, OCR, vectors, metadata, ACL tags.

### Why it matters
Bad indexes cause silent misses — the model never sees evidence.

### How it works
Store URI, text, caption, embedding(s), page, language, tenant_id; version the captioner.

### Intuition
If it isn't in metadata, you can't filter responsibly.

### Pitfalls
- No tenant_id
- Re-caption without version stamps
- Chunking that splits tables

### When to use
Corpora larger than one VLM context.


In [ ]:
# Demo 3: index + ACL
from dataclasses import dataclass, field
from typing import Any

@dataclass
class MMIndexRecord:
    doc_id: str; page: int; tenant_id: str; text: str; caption: str; vector: list[float]
    meta: dict[str, Any] = field(default_factory=dict)

class MiniIndex:
    def __init__(self): self.rows=[]
    def add(self,row): self.rows.append(row)
    def search(self, qv, tenant_id, k=3, meta_eq=None):
        def dot(a,b): return sum(x*y for x,y in zip(a,b))
        hits=[]
        for r in self.rows:
            if r.tenant_id!=tenant_id: continue
            if meta_eq and r.meta.get(meta_eq[0])!=meta_eq[1]: continue
            hits.append((dot(qv,r.vector), r))
        return sorted(hits, reverse=True)[:k]

idx=MiniIndex()
idx.add(MMIndexRecord("d1",1,"tA","Q3 18","revenue chart",[1,0,0],{"type":"chart"}))
idx.add(MMIndexRecord("d2",1,"tB","secret","x",[1,0,0],{"type":"chart"}))
idx.add(MMIndexRecord("d1",2,"tA","policy","text",[0.2,0.8,0],{"type":"text"}))
print([(s,r.doc_id,r.tenant_id) for s,r in idx.search([1,0,0],"tA", meta_eq=("type","chart"))])


In [ ]:
# Demo 4: config auditor
def audit(cfg):
    issues=[]
    if cfg.get("index")=="caption_only" and cfg.get("domain") in {"finance","engineering"}:
        issues.append("caption_only_risk_numeric_charts")
    if not cfg.get("attach_images_to_generator"): issues.append("retriever_without_images")
    if not cfg.get("tenant_field"): issues.append("missing_tenant_acl")
    if cfg.get("chunk_pages") and not cfg.get("keep_table_together"): issues.append("tables_may_split")
    return issues or ["ok"]
print(audit({"index":"caption_only","domain":"finance","attach_images_to_generator":False,"chunk_pages":True}))


In [ ]:
# Demo 5: rerank boost for numeric questions
def rerank(q, pages):
    numeric = any(w in q.lower() for w in ("how much","how many","revenue","total","%"))
    def score(p):
        base = len(set(q.lower().split()) & set((p.caption+" "+p.ocr).lower().split()))
        if numeric and any(ch.isdigit() for ch in p.ocr): base += 2
        return base
    return sorted(pages, key=score, reverse=True)
print([p.page_id for p in rerank("how much Q3 revenue", PAGES)])


In [ ]:
# Demo 6: citation formatter
def cite_answer(answer, page_ids):
    cites = " ".join(f"[{pid}]" for pid in page_ids)
    return f"{answer} {cites}".strip()
print(cite_answer("Q3 revenue was 18.", ["p1"]))


### Pitfalls

| Pitfall | Symptom | Mitigation |
|---------|---------|------------|
| Caption drift | Wrong retrieval | Version + OCR dual text |
| Layout loss | Bad table answers | Send page images; keep tables |
| Over-retrieve | Context bloat | Rerank; page budget |
| Under-retrieve | Confident nonsense | Hybrid search |
| ACL bugs | Cross-customer leaks | Mandatory tenant filters |


### Checklist — MM-RAG production

- [ ] Tenant filter in code path (not only query text)
- [ ] Images attached for visual questions
- [ ] Page budget enforced
- [ ] Captioner version stored
- [ ] Citation required in output schema


### Try it yourself — Multimodal RAG

1. Add metadata filters to a public search API wrapper.
2. Enforce max 3 images in pack().
3. Reject answers without citations.

**Stretch:** Sketch ColPali late-interaction in mermaid.


### Try it yourself — Eval

1. Build 10 questions with gold page_ids; measure recall@3.
2. Compare caption-only vs OCR+caption retrieval.


## Knowledge Check

**Q1.** Why attach images, not only OCR text?

<details><summary>Answer</summary>

Charts, layout, icons, and handwriting often aren't fully captured by OCR/captions.

</details>

**Q2.** Where must tenant ACL live?

<details><summary>Answer</summary>

In retrieval code filters — never only in the prompt.

</details>


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `multimodal RAG` | Retrieve visual/text evidence for generation |
| `late fusion` | Combine modality scores after retrieval |
| `captioner` | Image→text for indexing |
| `page budget` | Max pages attached to generator |
| `ACL` | Access-control / tenancy filters |


## Key Takeaways

- Retrieve and attach the right visual evidence
- Caption-only indexes lose chart/table fidelity
- Metadata + tenant ACL are mandatory
- Audit for drift, layout loss, and packing bugs


## Production Incident Patterns — multimodal RAG

| Symptom | Likely cause | First fix |
|---------|--------------|-----------|
| Sudden cost spike | `detail=high` on huge pages | Resize + tile budget |
| Fluent wrong fields | VLM hallucination | OCR hybrid + schema |
| Cross-customer leak | Missing tenant filter | ACL in retriever code |
| Flaky eval scores | Unfrozen prompts/models | Pin versions + bakeoff set |
| Latency SLO burn | Full-page high detail | Crop ROI → mini model |

```
ASCII control loop:
  ingest -> normalize -> route model -> generate -> validate -> (HITL|export)
                     ^                              |
                     +-------- metrics/audit <------+
```


In [ ]:
# Cross-cutting: redact secrets before logging multimodal payloads
import re, json

SECRET_RE = re.compile(r"(api[_-]?key|bearer\s+[A-Za-z0-9._\-]+)", re.I)

def safe_log(payload: dict) -> str:
    s = json.dumps(payload)
    s = SECRET_RE.sub("***", s)
    if "base64," in s:
        s = re.sub(r"base64,[A-Za-z0-9+/=]+", "base64,[REDACTED]", s)
    return s[:500]

print(safe_log({
    "model": "gpt-4o",
    "api_key": "YOUR_OPENAI_API_KEY",
    "content": "data:image/png;base64,AAAABBBBCCCC",
    "topic": "multimodal RAG",
}))


## Mini Case Study — multimodal RAG

**Scenario:** A team ships a vision feature in one week. Demo looks great on three happy-path images.
**Week 2:** finance reports wrong totals; legal asks about image retention; GPU/API bill 4× forecast.

**Retro questions**
1. What was the output contract (schema) on day one?
2. Which failure mode had no metric?
3. Was there a crop/detail budget?
4. Who owns HITL and appeals?

**Design rule:** if a field can move money or identity, it needs a validator + disagreement path before automation.


In [ ]:
# Cross-cutting: simple SLO helper for vision endpoints
from dataclasses import dataclass

@dataclass
class VisionSLO:
    availability: float = 0.995
    p95_ms: int = 4000
    max_critical_field_error_rate: float = 0.005

def breached(slo: VisionSLO, avail: float, p95: int, crit_err: float) -> list[str]:
    out = []
    if avail < slo.availability: out.append("availability")
    if p95 > slo.p95_ms: out.append("latency")
    if crit_err > slo.max_critical_field_error_rate: out.append("critical_accuracy")
    return out or ["ok"]

print("multimodal RAG", breached(VisionSLO(), 0.99, 5200, 0.02))


### Try it yourself — multimodal RAG ops

1. Write a one-page runbook section for on-call when multimodal RAG critical_accuracy SLO breaches.
2. Add a dashboard sketch: cost/1k images, CER/field error, HITL rate, p95 latency.
